# Lecture 03: Embeddings and PyTorch for LLMs

CS40008.01 · Baojian Zhou · Fudan University · September 23, 2026

**How do discrete tokens become trainable representations?**

Predict each result before running its cell. E01–E06 follow the slide order. These are **ungraded practices**, separate from Quiz 1 and Assignment A1. The complete notebook runs offline on CPU in the core `uv sync` environment; no model weights or tokenizer downloads are needed. The core examples and optional skip-gram corpus are **toy data**. Optional P01 uses a published co-occurrence table, cited there.

| Exercise | What you check |
| --- | --- |
| E01 · 4 min | Lookup shapes and one-hot equivalence |
| E02 · 4 min | One skip-gram gradient, then autograd |
| E03 · 4 min | Shifted targets, output projection, and stable cross-entropy |
| E04 · 6 min | A training loop on a compatible tiny batch |
| E05 · 4 min | Tied versus untied gradient paths |
| E06 · 4 min | Parameters, storage, and real model configurations |
| P01 · optional | PPMI from published counts |
| P02 · optional | Train and inspect toy skip-gram embeddings |
| P03 · optional | Verify a model row against its primary sources |

If you already have an older working copy, rename it before reopening the **Notebook** link; the launcher preserves existing student files. Run this notebook from top to bottom.

The 60-slide lecture includes a ten-slide classical bridge. The counts and SVD cells support that bridge; the main path remains next-token training, evaluation, and resource accounting.

---

<!-- bilingual-zh -->

# 第 3 讲：嵌入与用于大语言模型的 PyTorch
**离散词元如何变成可训练的表示？**运行前先预测结果。E01–E06 与课件顺序一致，是与测验 1 和作业 A1 分开的**不计分练习**。核心 `uv sync` 环境可在 CPU 上完整离线运行，无需下载权重或词元化器。核心示例和可选 skip-gram 语料是玩具数据；P01 使用所引已发表共现表。

E01 检查查表形状和独热等价；E02 计算 skip-gram 梯度与自动微分；E03 检查移位目标、输出投影和稳定交叉熵；E04 在兼容小批次上训练；E05 比较绑定与不绑定权重的梯度；E06 统计参数、存储和真实模型配置。P01–P03 分别为 PPMI、玩具 skip-gram 和核验模型资料的可选练习。若已有旧个人副本，重新打开 Notebook 链接前先重命名，启动器不会覆盖学生文件。自上而下运行。计数和 SVD 支持课件中的经典方法过渡；主线是下一词元训练、评估和资源核算。


In [ ]:
import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F

# CPU-only, reproducible small examples. Do not download a model.
torch.manual_seed(0)
V, D = 10, 4
print('PyTorch:', torch.__version__, '| vocabulary:', V, '| width:', D)

## E01 · Predict the lookup shapes

A batch of IDs has shape $(B,T)$. The table $E$ has shape $(|V|,d)$ and stores trainable parameters. Predict the shapes of `embedding.weight`, `token_ids`, `x`, and `one_hot` before running the cell.

We use contiguous toy IDs, so the number of embedding rows equals the vocabulary size. E06 will distinguish these quantities for real models.

---

<!-- bilingual-zh -->

## E01 · 预测查表形状
ID 批次形状为 $(B,T)$，可训练表 $E$ 的形状为 $(|V|,d)$。运行前预测 `embedding.weight`、`token_ids`、`x` 和 `one_hot` 的形状。玩具 ID 连续，因此嵌入行数等于词表大小；E06 会区分真实模型中的这两个量。


In [ ]:
embedding = torch.nn.Embedding(V, D)
token_ids = torch.tensor([[1, 5, 5], [7, 1, 0]])
x = embedding(token_ids)
one_hot = F.one_hot(token_ids, num_classes=V).float()
e01_shapes = {'weight': tuple(embedding.weight.shape), 'ids': tuple(token_ids.shape),
              'x': tuple(x.shape), 'one_hot': tuple(one_hot.shape)}
e01_same = torch.allclose(one_hot @ embedding.weight, x)
print(e01_shapes)
print('one-hot product equals lookup:', e01_same)
print('Repeated ID 5 gives the same row:', torch.equal(x[0, 1], x[0, 2]))

e01_dtypes = [str(token_ids.dtype), str(embedding.weight.dtype), str(x.dtype)]
print('ID / weight / output dtypes:', e01_dtypes)
print('ID / weight devices:', token_ids.device, embedding.weight.device)


**Check:** `(10, 4)`, `(2, 3)`, `(2, 3, 4)`, and `(2, 3, 10)`. The one-hot product is a mathematical explanation, not an efficient implementation. `nn.Embedding` selects rows without materializing the one-hot tensor.

The lookup for a token ID stays the same across contexts. A separate context network can transform that lookup into different contextual states.

---

<!-- bilingual-zh -->

**核对：**依次为 `(10,4)`、`(2,3)`、`(2,3,4)`、`(2,3,10)`。独热乘积用于数学解释，并非高效实现；`nn.Embedding` 直接选行，无需构造独热张量。同一词元 ID 在不同上下文中查到的向量相同，独立的上下文网络可进一步将其变为不同的上下文状态。


### Classical bridge: counts and compact vectors

These invented counts reproduce the slides: rows are **tea, coffee, car**; columns are **drink, hot, drive**. Compute PPMI relative to independence, then compress the 3-by-3 matrix to two dimensions with truncated SVD. The retained factors minimize squared reconstruction error at the chosen rank; their signs are not semantically meaningful.

This is a small demonstration, not another timed exercise. Longer count arithmetic remains optional P01. Sources: [Jurafsky and Martin, Appendix J](https://web.stanford.edu/~jurafsky/slp3/J.pdf); [Pennington et al. (2014), related work and GloVe objective](https://aclanthology.org/D14-1162/). GloVe and SVD use different objectives.

---

<!-- bilingual-zh -->

### 经典方法过渡：计数与紧凑向量
这些虚构计数与课件一致：行是 tea、coffee、car；列是 drink、hot、drive。根据独立性计算 PPMI，再用截断 SVD 将 3×3 矩阵压缩到两维。保留的因子在指定秩下最小化平方重建误差，其正负号本身没有语义意义。这是小演示，不是额外计时练习；较长的计数运算见可选 P01。来源见原文；GloVe 与 SVD 的目标函数不同。


In [ ]:
bridge_counts = torch.tensor([[8., 6., 0.], [7., 5., 0.], [0., 0., 9.]], dtype=torch.float64)
bridge_total = bridge_counts.sum()
bridge_rows = bridge_counts.sum(dim=1, keepdim=True)
bridge_columns = bridge_counts.sum(dim=0, keepdim=True)
bridge_pmi = torch.log2(bridge_counts * bridge_total / (bridge_rows * bridge_columns))
bridge_ppmi = bridge_pmi.clamp(min=0)
bridge_u, bridge_s, bridge_vh = torch.linalg.svd(bridge_ppmi, full_matrices=False)
bridge_vectors = bridge_u[:, :2] * bridge_s[:2]
bridge_reconstruction = bridge_vectors @ bridge_vh[:2]
print('total:', bridge_total.item(), '| PMI(tea, drink):', bridge_pmi[0, 0].item(), 'bits')
print('PPMI matrix:', bridge_ppmi.tolist())
print('word vectors:', tuple(bridge_vectors.shape), '| reconstructed matrix:', tuple(bridge_reconstruction.shape))
print('rank-2 reconstruction error:', torch.linalg.norm(bridge_ppmi - bridge_reconstruction).item())

**Check:** the total is 35 and PMI(tea, drink) is $\log_2(4/3)=0.4150$ bits. Zero counts map from negative infinity to PPMI zero. Each row of the 3-by-2 factor is a static word vector.

The lecture then compares CBOW and skip-gram, performs E02, and introduces fastText's overlapping character n-grams. [Bojanowski et al. (2017), Section 3.2](https://aclanthology.org/Q17-1010/) explains the subword construction. These methods supply ideas and baselines; an LLM's contextual state also depends on a learned context network.

---

<!-- bilingual-zh -->

**核对：**总数为 35，PMI(tea, drink)=$\log_2(4/3)=0.4150$ 比特。零计数对应负无穷 PMI，PPMI 将其置零。3×2 因子的每一行是静态词向量。课件随后比较 CBOW 与 skip-gram、完成 E02，并介绍 fastText 的重叠字符 n-gram。原文链接说明子词构造；这些方法提供思路和基线，而 LLM 的上下文状态还依赖学习得到的上下文网络。


## E02 · Predict one gradient

For one observed pair and one sampled negative pair:

$$\ell=-\log\sigma(\mathbf w^\top\mathbf c_+)-\log\sigma(-\mathbf w^\top\mathbf c_-)$$

Use $\mathbf w=(1,0.5)$, $\mathbf c_+=(0.5,1)$, $\mathbf c_-=(1,-1)$. Given $\sigma(1)=0.7311$ and $\sigma(0.5)=0.6225$, compute only $\nabla_{\mathbf c_+}\ell=(\sigma(\mathbf w^\top\mathbf c_+)-1)\mathbf w$ by hand. Then check all gradients with autograd.

The sigmoid scores observed versus sampled pairs. It is **not** a normalized next-token distribution. See [Mikolov et al. (2013), §2.2](https://papers.nips.cc/paper_files/paper/2013/file/9aa42b31882ec039965f3c4923ce901b-Paper.pdf).

---

<!-- bilingual-zh -->

## E02 · 预测一个梯度
对一个观察到的正样本对和一个采样负样本对，使用上方损失公式。取 $w=(1,0.5)$、正上下文 $(0.5,1)$、负上下文 $(1,-1)$，已知 $\sigma(1)=0.7311$ 和 $\sigma(0.5)=0.6225$。先手算题目所示的正上下文梯度，再用 autograd 核验所有梯度。Sigmoid 对观察对与采样对打分，**不是**归一化的下一词元分布。


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


W_VEC, C_POS, C_NEG, ETA = [1.0, 0.5], [0.5, 1.0], [1.0, -1.0], 0.5

s_pos = sigmoid(dot(W_VEC, C_POS))
s_neg = sigmoid(dot(W_VEC, C_NEG))
e02_loss = -math.log(s_pos) - math.log(1.0 - s_neg)  # sigma(-z) = 1 - sigma(z)

e02_grads = {
    "c_pos": [(s_pos - 1.0) * x for x in W_VEC],
    "c_neg": [s_neg * x for x in W_VEC],
    "w": [(s_pos - 1.0) * p + s_neg * n for p, n in zip(C_POS, C_NEG)],
}
e02_updated = {
    "c_pos": [v - ETA * g for v, g in zip(C_POS, e02_grads["c_pos"])],
    "c_neg": [v - ETA * g for v, g in zip(C_NEG, e02_grads["c_neg"])],
    "w": [v - ETA * g for v, g in zip(W_VEC, e02_grads["w"])],
}

print(f"scores: w.c_pos = {dot(W_VEC, C_POS):.1f}, w.c_neg = {dot(W_VEC, C_NEG):.1f}")
print(f"s_pos = {s_pos:.4f}, s_neg = {s_neg:.4f}, loss = {e02_loss:.4f}")
for name in ["c_pos", "c_neg", "w"]:
    grad = ", ".join(f"{g:+.4f}" for g in e02_grads[name])
    new = ", ".join(f"{v:+.4f}" for v in e02_updated[name])
    print(f"grad {name:5s} = ({grad})   updated {name:5s} = ({new})")

after = -math.log(sigmoid(dot(e02_updated["w"], e02_updated["c_pos"]))) - math.log(1.0 - sigmoid(dot(e02_updated["w"], e02_updated["c_neg"])))
print(f"loss after one step = {after:.4f}")

**Check:** the scores are $1.0$ and $0.5$, the loss is $1.2873$, and the requested gradient is $(-0.2689,-0.1345)$. With the center vector fixed, an SGD step increases its dot product with the positive context. The code above additionally updates all three vectors and checks the resulting loss.

The scalar `math.log` expressions above illustrate these small, safe numbers. For trainable tensors use the stable `F.logsigmoid` implementation below.

---

<!-- bilingual-zh -->

**核对：**两个分数为 1.0 和 0.5，损失为 1.2873，所求梯度为 $(-0.2689,-0.1345)$。固定中心向量时，SGD 一步会增大它与正上下文的点积。上方代码还更新三个向量并检查更新后的损失。`math.log` 标量表达式仅用于数值安全的小例子；可训练张量应使用下方稳定的 `F.logsigmoid`。


In [ ]:
import torch
import torch.nn.functional as F

w = torch.tensor(W_VEC, requires_grad=True)
c_pos = torch.tensor(C_POS, requires_grad=True)
c_neg = torch.tensor(C_NEG, requires_grad=True)

loss = -F.logsigmoid(w @ c_pos) - F.logsigmoid(-(w @ c_neg))
loss.backward()

e02_autograd = {"c_pos": c_pos.grad.tolist(), "c_neg": c_neg.grad.tolist(), "w": w.grad.tolist()}
print(f"loss = {loss.item():.4f}")
for name, grad in e02_autograd.items():
    print(f"autograd {name:5s} =", [round(g, 4) for g in grad], "| by hand =", [round(g, 4) for g in e02_grads[name]])

## E03 · Check the targets and loss shape

The toy sequences are `[1,2,3,4,5]` and `[5,6,7,8,9]`. Slice each sequence into inputs and next-token targets. Predict the last target in each row, the embedding/logit shapes, and the number of predictions in the mean loss.

This data is constructed for the lecture. It does not use the assignment's data or code.

---

<!-- bilingual-zh -->

## E03 · 检查目标与损失形状
玩具序列为 `[1,2,3,4,5]` 和 `[5,6,7,8,9]`。将每个序列切为输入和下一词元目标。预测每行最后的目标、嵌入与 logits 形状，以及平均损失包含的预测数。这是课堂构造数据，不使用作业数据或代码。


In [ ]:
tokens = torch.tensor([[1, 2, 3, 4, 5], [5, 6, 7, 8, 9]])
inputs = tokens[:, :-1]
targets = tokens[:, 1:]
B, T = inputs.shape
print('inputs :', inputs.tolist())
print('targets:', targets.tolist())
print('B, T:', B, T, '| number of predictions:', B * T)
print('targets contiguous:', targets.is_contiguous())

e03_flat_targets = targets.reshape(-1)
try:
    targets.view(-1)
    e03_view_failed = False
except RuntimeError:
    e03_view_failed = True
print('view failed on this slice:', e03_view_failed)
print('reshape preserves label order:', e03_flat_targets.tolist())


### The model used today

`TinyLM` is a **bigram model with a learned vector bottleneck**: $h_t=E[w_t]$, then $z_t=h_tW_{out}^{\top}$. It cannot distinguish two prefixes that end in the same token ID. Next week we add a context network.

Initialize the output table to zero for an exactly uniform first prediction. Keep the input table nonzero so the output weights can learn. Initializing *both* tables to zero would stall learning through this product. The `share_weights` option is used later in E05; the main training example is untied.

---

<!-- bilingual-zh -->

### 今天使用的模型
`TinyLM` 是带**可学习向量瓶颈的二元模型**：先查找当前词元嵌入，再做输出投影。它不能区分以同一 ID 结尾的两个前缀；下周加入上下文网络。输出表初始化为零，使首次预测严格均匀；输入表保持非零，以便输出权重学习。若两张表都初始化为零，通过乘积的学习会停滞。E05 使用 `share_weights`；主要训练示例不绑定权重。


In [ ]:
class TinyLM(torch.nn.Module):
    def __init__(self, vocab_size, dim, share_weights=False, zero_output=True):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, dim)
        self.output = torch.nn.Linear(dim, vocab_size, bias=False)
        torch.nn.init.normal_(self.embedding.weight, std=0.02)
        if share_weights:
            self.output.weight = self.embedding.weight
        elif zero_output:
            torch.nn.init.zeros_(self.output.weight)
        else:
            torch.nn.init.normal_(self.output.weight, std=0.02)

    def forward(self, ids):
        return self.output(self.embedding(ids))


def next_token_loss(logits, labels):
    # reshape also works for the non-contiguous target slice above.
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


torch.manual_seed(0)
model = TinyLM(V, D)
logits = model(inputs)
loss = next_token_loss(logits, targets)
manual_loss = -F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).mean()
e03_shapes = {'embeddings': tuple(model.embedding(inputs).shape), 'logits': tuple(logits.shape)}
e03_uniform_loss = loss.item()
e03_manual_loss = manual_loss.item()
print(e03_shapes)
print('CE:', e03_uniform_loss, '| direct NLL:', e03_manual_loss, '| log V:', math.log(V))

**Check:** final targets $5$ and $9$, embeddings $(2,4,4)$, logits $(2,4,10)$, and eight predictions. Zero logits give loss $\log 10=2.3026$ nats and perplexity $10$.

`F.cross_entropy` takes **raw logits** and integer target IDs. It already applies log-softmax. Do not apply softmax first. Arbitrary random initialization is not guaranteed to produce this uniform baseline: its loss depends on the scale of the logits.

### Finite precision

Before running the next cell, predict what happens when float32 `sigmoid(100)` rounds to one. These two loss formulas are mathematically equivalent.

---

<!-- bilingual-zh -->

**核对：**最终目标为 5 和 9；嵌入形状 `(2,4,4)`，logits 形状 `(2,4,10)`，共 8 个预测。全零 logits 的损失为 $\log 10=2.3026$ nats，困惑度为 10。`F.cross_entropy` 接收**原始 logits** 和整数目标 ID，内部已计算 log-softmax，不能先做 softmax。任意随机初始化不保证均匀基线，损失取决于 logits 的尺度。

### 有限精度
运行下一单元前，预测 float32 的 `sigmoid(100)` 四舍五入为 1 时会发生什么。两个损失公式在数学上等价。


In [ ]:
stability = {}
for score in [1.0, 100.0]:
    s = torch.tensor(score, dtype=torch.float32)
    literal = -(1 - torch.sigmoid(s)).log()
    stable = -F.logsigmoid(-s)
    stability[score] = {'literal': literal.item(), 'stable': stable.item()}
    print(score, stability[score])

## E04 · Train on one tiny batch

Run 200 SGD updates on the shifted batch. Record the loss before any update and after the last update, then compare all eight predictions with the targets. Why is this not a test of generalization?

This batch is compatible with the bigram model: each observed input ID has one next-token label. A batch that assigns different next tokens to the same input cannot be fit perfectly by this model. Even fitting a compatible batch checks only the mechanics, not performance on new text.

---

<!-- bilingual-zh -->

## E04 · 在一个极小批次上训练
对移位后的批次进行 200 次 SGD 更新。记录首次更新前和最后更新后的损失，将 8 个预测与目标比较。为什么这不是泛化测试？该批次与二元模型兼容：每个观测输入 ID 都有唯一下一词元标签。若同一输入对应不同后继，模型不能完全拟合。即使拟合兼容批次，也只检查机制，不说明新文本上的表现。


In [ ]:
torch.manual_seed(0)
model = TinyLM(V, D)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
e04_losses = [next_token_loss(model(inputs), targets).item()]
e04_initial_embedding = model.embedding.weight.detach().clone()
for step in range(200):
    optimizer.zero_grad()
    loss = next_token_loss(model(inputs), targets)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        e04_losses.append(next_token_loss(model(inputs), targets).item())
    if step in (0, 9, 49, 199):
        print(f'after {step + 1:3d} updates: loss = {e04_losses[-1]:.4f}')

with torch.no_grad():
    e04_predictions = model(inputs).argmax(-1)
print('initial loss:', e04_losses[0])
print('predictions:', e04_predictions.tolist())
print('targets    :', targets.tolist())

**Check:** initial loss is 2.3026 nats, final loss is below 0.1, and predictions match all eight targets. Exact losses can differ slightly across PyTorch versions. The training loop clears old gradients, runs a forward pass, computes loss, backpropagates, and takes an optimizer step.

With zero output weights, the input embedding's gradient is zero on the *first* update. The output weights change first; subsequent updates can change the input table. A nonzero gradient is not guaranteed merely because a parameter participates in the model.

The slides' loss curve samples this run at updates 0, 1, 5, 10, 20, 50, 100, 150, and 200. It is a teaching figure, not a benchmark. To evaluate generalization, reserve text that did not participate in training.

---

<!-- bilingual-zh -->

**核对：**初始损失为 2.3026 nats，最终低于 0.1，8 个预测均匹配目标。PyTorch 版本不同可能有细微差异。训练循环依次清除旧梯度、前向计算、计算损失、反向传播、优化器更新。

输出权重为零时，输入嵌入在**第一次**更新中的梯度为零；输出权重先变化，之后输入表才可能变化。参数参与模型不保证梯度非零。课件损失曲线采样第 0、1、5、10、20、50、100、150、200 次更新，是教学图而非基准。评估泛化时要保留未参与训练的文本。


### Evaluate two held-out coverage probes

Pairs **0 → 1** and **9 → 0** never participate in optimization. Their input IDs are absent from the eight training pairs. They are deliberately selected synthetic coverage probes, not an independent sample of realistic text and not a reliable estimate of corpus generalization.

Predict whether their untied input-table rows changed during training. Then inspect the two losses. `model.eval()` controls modules such as dropout; `torch.no_grad()` disables gradient recording. This particular TinyLM has no dropout or batch normalization.

---

<!-- bilingual-zh -->

### 评估两个留出覆盖探针
`0 → 1` 和 `9 → 0` 从未参与优化，其输入 ID 也不在八个训练对中。它们是刻意选择的合成覆盖探针，不是现实文本的独立样本，也不能可靠估计语料泛化。先预测不绑定输入表的相应行是否变化，再观察两个损失。`model.eval()` 控制 dropout 等模块；`torch.no_grad()` 禁用梯度记录。本 TinyLM 没有 dropout 或批归一化。


In [ ]:
val_inputs = torch.tensor([[0, 9]])
val_targets = torch.tensor([[1, 0]])
e04_eval_parameters_before = [p.detach().clone() for p in model.parameters()]
e04_eval_gradients_before = [p.grad.detach().clone() for p in model.parameters()]
e04_previous_mode = model.training
model.eval()
with torch.no_grad():
    val_logits = model(val_inputs)
    val_loss = next_token_loss(val_logits, val_targets)
    e04_validation_loss = val_loss.item()
    e04_validation_requires_grad = val_logits.requires_grad
model.train(e04_previous_mode)
e04_eval_parameters_after = [p.detach().clone() for p in model.parameters()]
e04_eval_gradients_after = [p.grad.detach().clone() for p in model.parameters()]
e04_mode_restored = model.training == e04_previous_mode
print('training loss:', e04_losses[-1], '| two-pair probe loss:', e04_validation_loss)
print('probe input IDs:', val_inputs.tolist(), '| probe targets:', val_targets.tolist())
print('probe lookup rows unchanged:', torch.equal(model.embedding.weight.detach()[[0, 9]], e04_initial_embedding[[0, 9]]))

**Interpretation:** fitting the training batch does not teach the lookup table how to represent input IDs it never sees. Their predictions can still change through updates to the output table. Evaluation performs no parameter or gradient update and restores the previous training mode.

The two constructed pairs probe a coverage gap. Assessing language-model generalization requires a sufficiently large, representative held-out corpus; reserve a separate final test set when using validation to make training decisions.

---

<!-- bilingual-zh -->

**解释：**拟合训练批次不会教会查表层如何表示从未见过的输入 ID；但输出表更新仍可能改变这些输入的预测。评估不更新参数或梯度，并恢复之前的训练模式。两个构造对只探测覆盖缺口。评估语言模型泛化需要足够大且有代表性的留出语料；用验证集决定训练方案时，还应保留独立最终测试集。


## E05 · Which rows receive gradients?

For inputs `[[1,5,5],[7,1,0]]`, which input embedding rows receive loss gradients?

Compare **two equal but independent tables** with **one shared parameter**. We initialize both cases to the same values so their logits agree. The untied output table is nonzero. The labels below illustrate gradients only; this is not the E04 memorization batch.

Weight tying must refer to the same `Parameter`, not merely copy its values once. Tie before constructing an optimizer. [Press and Wolf (2017), §3](https://aclanthology.org/E17-2025/) discusses the different updates.

---

<!-- bilingual-zh -->

## E05 · 哪些行收到梯度？
输入 `[[1,5,5],[7,1,0]]` 时，哪些输入嵌入行收到损失梯度？比较**两个数值相同但独立的表**与**一个共享参数**；两种情况初始化相同以保证 logits 一致，未绑定的输出表非零。下方标签只用于说明梯度，不是 E04 的记忆批次。权重绑定必须引用同一个 `Parameter`，不能只是复制一次数值；应在构造优化器前绑定。


In [ ]:
torch.manual_seed(7)
grad_ids = torch.tensor([[1, 5, 5], [7, 1, 0]])
grad_targets = torch.tensor([[5, 5, 2], [1, 0, 3]])
untied = TinyLM(V, D, zero_output=False)
tied = TinyLM(V, D, share_weights=True)
with torch.no_grad():
    untied.output.weight.copy_(untied.embedding.weight)
    tied.embedding.weight.copy_(untied.embedding.weight)


def active_rows(gradient):
    return (gradient.abs().sum(-1) > 0).nonzero().flatten().tolist()


e05_same_logits = torch.allclose(untied(grad_ids), tied(grad_ids))
for name, candidate in [('untied', untied), ('tied', tied)]:
    candidate.zero_grad()
    next_token_loss(candidate(grad_ids), grad_targets).backward()
    print(name, 'input-table gradient rows:', active_rows(candidate.embedding.weight.grad))

e05_untied_rows = active_rows(untied.embedding.weight.grad)
e05_tied_rows = active_rows(tied.embedding.weight.grad)
e05_gradient_sum = torch.allclose(
    tied.embedding.weight.grad,
    untied.embedding.weight.grad + untied.output.weight.grad,
    atol=1e-7,
)
print('identical logits:', e05_same_logits)
print('shared gradient = input contribution + output contribution:', e05_gradient_sum)
print('same Parameter object:', tied.output.weight is tied.embedding.weight)

**Check:** the untied input table has gradients only in rows 0, 1, 5, and 7. This example gives all ten rows nonzero gradients when tied. The shared gradient is the sum of the two pathways in the untied calculation.

At one position the output-row gradient is $(p(j)-\mathbf{1}[j=y])h$. Every row participates in softmax, including rows absent from the input. A row *can* receive a gradient; special parameter values or cancellation can still make it zero. These statements concern the loss gradient. Weight decay or previous optimizer state can change a parameter even without a new nonzero lookup gradient.

---

<!-- bilingual-zh -->

**核对：**未绑定输入表只有第 0、1、5、7 行有梯度；此示例绑定时 10 行均非零。共享梯度等于未绑定计算中两条路径的梯度之和。每个位置的输出行梯度为 $(p(j)-\mathbf{1}[j=y])h$；所有行参与 softmax，包括未出现在输入中的行。一行*可能*收到梯度，但特殊参数或相互抵消仍可使其为零。这些讨论指损失梯度；即便没有新的非零查表梯度，权重衰减或已有优化器状态也可能改变参数。


## E06 · Count parameters and storage

For $M=32{,}000$ allocated rows and $d=512$, count the input/output parameters with and without tying, and the bf16 storage for one table. No bias or context-network parameters are included. One MiB is $2^{20}$ bytes.

Then compare two official Qwen configurations. `config.vocab_size` gives the allocated table rows $M$, which can exceed the number of token IDs in the tokenizer artifact. `hidden_size` gives the width in these models. The supplied metadata contains pinned source URLs and hashes; we read it locally, without downloading weights.

---

<!-- bilingual-zh -->

## E06 · 统计参数与存储
给定分配行数 $M=32,000$、宽度 $d=512$，计算绑定与不绑定时的输入／输出参数量，以及一张表的 bf16 存储；不计偏置或上下文网络。1 MiB 为 $2^{20}$ 字节。再比较两个官方 Qwen 配置。`config.vocab_size` 给出分配的表行数 $M$，可能大于词元化器文件中的 ID 数；这些模型的 `hidden_size` 给出宽度。提供的元数据固定了来源 URL 和哈希，代码本地读取，无需下载权重。


In [ ]:
e06_toy_counts = {'untied': count_parameters(untied), 'tied': count_parameters(tied)}
M, width = 32_000, 512
e06_exercise = {'one_table': M * width, 'untied': 2 * M * width,
                'bf16_bytes': M * width * 2, 'bf16_mib': M * width * 2 / 2**20}
print('toy model:', e06_toy_counts)
print('exercise:', e06_exercise)

In [ ]:
# The launcher copies assets beside your working notebook. The second path
# also supports running all cells from the repository root.
asset_candidates = [Path('assets/model-configs.json'),
                    Path('slides/lecture-03/assets/model-configs.json')]
model_asset = next((path for path in asset_candidates if path.is_file()), None)
if model_asset is None:
    raise FileNotFoundError('Open this notebook with its assets directory using the course launcher.')
model_evidence = json.loads(model_asset.read_text())
e06_models = {}
for cfg in model_evidence['models']:
    one_table = cfg['embedding_rows'] * cfg['hidden_size']
    unique = one_table * (1 if cfg['tie_word_embeddings'] else 2)
    result = {'one_table': one_table, 'unique_parameters': unique, 'bf16_mib': unique * 2 / 2**20}
    e06_models[cfg['model']] = result
    print(cfg['model'], '| tokenizer IDs:', cfg['tokenizer_entries'],
          '| embedding rows:', cfg['embedding_rows'], '| tied:', cfg['tie_word_embeddings'])
    print(result)

logit_B, logit_T, logit_M = 2, 1024, 151_936
e06_logit_mib = logit_B * logit_T * logit_M * 2 / 2**20
print('materialized bf16 logits:', e06_logit_mib, 'MiB')

**Check:** one exercise table has 16,384,000 parameters and occupies 31.25 MiB in bf16. Untied input/output tables have twice as many parameters. The toy models have 80 untied parameters or 40 tied parameters.

| Model | One table | Unique input + output parameters | bf16 storage |
| --- | ---: | ---: | ---: |
| Qwen3-0.6B | 155,582,464 | 155,582,464 | 296.75 MiB |
| Qwen3-8B | 622,329,856 | 1,244,659,712 | 2,374 MiB |

Both pinned tokenizer files contain 151,669 distinct IDs (maximum 151,668), but each configuration allocates 151,936 rows. Count the union of IDs in `model.vocab` and `added_tokens`; simply adding the lengths could count the same ID twice. An embedding table must accommodate the maximum supported ID, even if an ID space has gaps.

Training also requires gradients, optimizer state, and activations. A materialized `(2,1024,151936)` bf16 logits tensor alone occupies 593.5 MiB. This is not a total-memory estimate; precision, kernels and intermediate materialization matter. Holding $B,T,d$ fixed, dense output projection takes approximately $2BTdM$ forward FLOPs, counting a multiply and an add separately. Weight tying does not remove this computation.

---

<!-- bilingual-zh -->

**核对：**练习的一张表有 16,384,000 个参数，bf16 占 31.25 MiB；不绑定的输入／输出表参数量翻倍。玩具模型不绑定时 80 个参数，绑定时 40 个。Qwen3-0.6B 单表及唯一输入／输出参数均为 155,582,464，bf16 为 296.75 MiB；Qwen3-8B 单表 622,329,856，唯一输入／输出参数为 1,244,659,712，bf16 为 2,374 MiB。

两个固定版本的词元化器均有 151,669 个不同 ID（最大值 151,668），配置却各分配 151,936 行。应对 `model.vocab` 和 `added_tokens` 中 ID 取并集；直接相加长度可能重复计数。即使 ID 空间有间隙，嵌入表也必须容纳最大支持 ID。

训练还需梯度、优化器状态与激活。仅将 `(2,1024,151936)` 的 bf16 logits 实体化就占 593.5 MiB；这不是总内存估计，精度、内核和中间张量是否实体化都重要。固定 $B,T,d$，稠密输出投影前向约需 $2BTdM$ FLOPs（乘加分别计数）；权重绑定不会消除这部分计算。


### Training memory: a formula and a measurement

For the exercise's $P=16{,}384{,}000$ unique parameters, assume parameters, gradients, and Adam's two moment tensors are all fp32. Their combined payload is $16P$ bytes, or 250 MiB. This excludes activations, temporary buffers, scalar counters, and allocator overhead. SGD without momentum needs no persistent moment tensors.

Inspect a separate 80-parameter CPU model after one Adam step, when its optimizer state has been initialized. This leaves the E04 trained model unchanged. Source: [CS336 Lecture 2, optimizer](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py).

---

<!-- bilingual-zh -->

### 训练内存：公式与测量
练习有 $P=16,384,000$ 个唯一参数。假定参数、梯度及 Adam 两个动量张量均为 fp32，总数据量为 $16P$ 字节，即 250 MiB；不含激活、临时缓冲区、标量计数器和分配器开销。不带动量的 SGD 不需要持久动量张量。对另一个 80 参数 CPU 模型执行一次 Adam 更新后检查其已初始化的优化器状态，不影响 E04 已训练模型。


In [ ]:
P = e06_exercise['one_table']
e06_training_mib = {'parameters': P * 4 / 2**20, 'gradients': P * 4 / 2**20,
                    'adam_moments': 2 * P * 4 / 2**20, 'subtotal': 4 * P * 4 / 2**20}
probe = TinyLM(V, D)
probe_opt = torch.optim.Adam(probe.parameters(), lr=0.01, foreach=False)
probe_opt.zero_grad()
next_token_loss(probe(inputs), targets).backward()
probe_opt.step()
nbytes = lambda t: t.numel() * t.element_size()
parameter_bytes = sum(nbytes(p) for p in probe.parameters())
gradient_bytes = sum(nbytes(p.grad) for p in probe.parameters())
state_bytes = sum(nbytes(t) for state in probe_opt.state.values()
                  for t in state.values() if torch.is_tensor(t))
moment_bytes = sum(nbytes(state[key]) for state in probe_opt.state.values()
                   for key in ['exp_avg', 'exp_avg_sq'])
e06_measured_bytes = {'parameters': parameter_bytes, 'gradients': gradient_bytes,
                      'optimizer_state': state_bytes, 'moments': moment_bytes,
                      'scalar_counters': state_bytes - moment_bytes}
e06_projection = {'parameters': D * V, 'positions': B * T, 'forward_flops': 2 * B * T * D * V}
print('large-table fp32 ledger, MiB:', e06_training_mib)
print('tiny-model measured payload, bytes:', e06_measured_bytes)
print('toy forward output projection:', e06_projection)

**Check:** 62.5 MiB for parameters, 62.5 MiB for gradients, and 125 MiB for Adam's two moments give 250 MiB before the additional costs. The small CPU model has 320 parameter bytes, 320 gradient bytes, and 640 moment bytes; optimizer state also contains scalar step counters. These payloads are not peak device-memory measurements.

The toy output table contains 40 parameters and is used at eight positions. Counting a multiply-add as two FLOPs gives approximately 640 forward projection FLOPs. This excludes the loss, backward pass, and optimizer update.

---

<!-- bilingual-zh -->

**核对：**参数 62.5 MiB、梯度 62.5 MiB、Adam 两个动量 125 MiB，合计 250 MiB，尚不含其他成本。小型 CPU 模型分别有 320、320、640 字节；优化器状态另有标量步数计数器。这些数据量不是设备峰值内存。玩具输出表有 40 个参数，在 8 个位置使用；乘加计为两次 FLOP，前向投影约 640 FLOPs，不含损失、反向传播和优化器更新。


## P01 · PPMI from a co-occurrence table (optional)

The counts below are from Jurafsky and Martin, [Appendix J, Figure J.2](https://web.stanford.edu/~jurafsky/slp3/J.pdf). They are published word–context counts, not the invented counts in the slide illustration.

$$\mathrm{PMI}(w,c)=\log_2\frac{p(w,c)}{p(w)p(c)},\qquad \mathrm{PPMI}(w,c)=\max(\mathrm{PMI}(w,c),0)$$

Use the totals to compute PPMI(information, data), then compare a rare but associated pair such as cherry/pie. This historical background is optional; it is not needed to run a neural LM training step.

---

<!-- bilingual-zh -->

## P01 · 根据共现表计算 PPMI（可选）
下方计数来自 Jurafsky 和 Martin 附录 J 图 J.2，是已发表的词与上下文共现计数，不是课件示意图中的虚构计数。根据上方 PMI、PPMI 公式及总数计算 PPMI(information, data)，再比较如 cherry/pie 这样罕见但相关的组合。这是可选历史背景，不是运行神经语言模型训练步骤的前提。


In [ ]:
import math

CONTEXTS = ["computer", "data", "result", "pie", "sugar"]
COUNTS = {
    "cherry":      [2, 8, 9, 442, 25],
    "strawberry":  [0, 0, 1, 60, 19],
    "digital":     [1670, 1683, 85, 5, 4],
    "information": [3325, 3982, 378, 5, 13],
}

TOTAL = sum(sum(row) for row in COUNTS.values())
ROW_TOTAL = {word: sum(row) for word, row in COUNTS.items()}
COLUMN_TOTAL = {c: sum(COUNTS[w][j] for w in COUNTS) for j, c in enumerate(CONTEXTS)}


def pmi(word, context):
    # Pointwise mutual information in bits; -inf when the pair never co-occurs.
    joint = COUNTS[word][CONTEXTS.index(context)] / TOTAL
    if joint == 0:
        return float("-inf")
    return math.log2(joint / ((ROW_TOTAL[word] / TOTAL) * (COLUMN_TOTAL[context] / TOTAL)))


def ppmi(word, context):
    return max(pmi(word, context), 0.0)


print("grand total:", TOTAL, "| row totals:", ROW_TOTAL, "| column totals:", COLUMN_TOTAL)
for word, context in [("information", "data"), ("cherry", "pie"), ("strawberry", "computer"), ("digital", "pie")]:
    print(f"PMI({word}, {context}) = {pmi(word, context):8.4f}   PPMI = {ppmi(word, context):.4f}")

**Check:** $p(\text{information},\text{data}) = 3982/11716 = 0.3399$, $p(\text{information}) = 7703/11716 = 0.6575$, $p(\text{data}) = 5673/11716 = 0.4842$, so $\mathrm{PMI} = \log_2(0.3399/(0.6575 \times 0.4842)) = 0.0944$ bits. The pair is frequent, yet barely more frequent than chance, because both words are frequent. `cherry` and `pie` are rarer but strongly associated ($4.38$ bits). A zero count gives $-\infty$, and a negative PMI needs a large corpus to be reliable: PPMI sets both to $0$.

---

<!-- bilingual-zh -->

**核对：**联合概率 $3982/11716=0.3399$，information 边缘概率 $7703/11716=0.6575$，data 边缘概率 $5673/11716=0.4842$，因此 PMI 为 0.0944 比特。这一对虽然频繁，但两个词各自都很常见，因此只比独立情况下略频繁。`cherry` 与 `pie` 较少见却强相关（4.38 比特）。零计数产生负无穷；负 PMI 需要大语料才可靠；PPMI 将两者均设为 0。


## P02 · Train and inspect toy skip-gram vectors (optional)

Two tables score observed and sampled pairs, as in E02. This larger toy uses window radius 3, five negative samples per pair, and a noise distribution proportional to frequency raised to 0.75. The corpus is generated from templates with deliberately distinct contexts for people, animals, foods, and places. These patterns are designed examples, not findings about language.

This practice takes a few seconds on CPU. It is outside the class timing. Neighbors and analogy results on this small corpus can be fragile; interpret them as a way to inspect the learned representation.

---

<!-- bilingual-zh -->

## P02 · 训练并检查玩具 skip-gram 向量（可选）
与 E02 一样，用两张表给观察对及采样对打分。这个稍大的玩具示例使用窗口半径 3、每对 5 个负样本，噪声分布与频率的 0.75 次幂成正比。模板语料刻意为人、动物、食物和地点构造不同上下文；这些是设计出的例子，而非关于语言的研究发现。CPU 上运行几秒，课堂计时之外完成。小语料上的邻居和类比结果可能不稳定，应将其作为检查已学表示的方法。


In [ ]:
import random


def build_toy_corpus(sentences_per_template=150, seed=2026):
    rng = random.Random(seed)
    people = {
        "king": ("he", "rules", "palace"), "queen": ("she", "rules", "palace"),
        "prince": ("he", "rules", "palace"), "princess": ("she", "rules", "palace"),
        "man": ("he", "works", "village"), "woman": ("she", "works", "village"),
        "boy": ("he", "works", "village"), "girl": ("she", "works", "village"),
    }
    animals, foods = ["cat", "dog", "horse", "bird"], ["rice", "bread", "fish", "apple"]
    capitals = {"beijing": "china", "paris": "france", "london": "england", "tokyo": "japan"}
    sentences = []
    for _ in range(sentences_per_template):
        person = rng.choice(list(people))
        pronoun, verb, place = people[person]
        age = "young " if person in ("prince", "princess", "boy", "girl") else ""
        sentences.append(f"the {age}{person} said {pronoun} {verb} in the {place}")
        sentences.append(f"the {age}{person} eats {rng.choice(foods)} in the {place}")
        sentences.append(f"the {rng.choice(animals)} chases the {rng.choice(animals)} and eats {rng.choice(foods)}")
        capital = rng.choice(list(capitals))
        sentences.append(f"{capital} is the capital city of {capitals[capital]}")
        sentences.append(f"the {person} travels from the {place} to {rng.choice(list(capitals))}")
    return [s.split() for s in sentences]


TOY_SENTENCES = build_toy_corpus()
VOCAB = sorted({word for sentence in TOY_SENTENCES for word in sentence})
WORD_TO_ID = {word: i for i, word in enumerate(VOCAB)}
print(len(TOY_SENTENCES), "sentences,", sum(map(len, TOY_SENTENCES)), "tokens,", len(VOCAB), "word types")
print(" ".join(TOY_SENTENCES[0]))

In [ ]:
def positive_pairs(sentences, window=3):
    pairs = []
    for sentence in sentences:
        ids = [WORD_TO_ID[word] for word in sentence]
        for i, center in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i:
                    pairs.append((center, ids[j]))
    return torch.tensor(pairs)


PAIRS = positive_pairs(TOY_SENTENCES)
counts = torch.bincount(torch.tensor([WORD_TO_ID[w] for s in TOY_SENTENCES for w in s]), minlength=len(VOCAB)).float()
NOISE = counts.pow(0.75) / counts.pow(0.75).sum()       # P(w) proportional to count^0.75
print("positive pairs:", tuple(PAIRS.shape), "| first pairs:", [(VOCAB[a], VOCAB[b]) for a, b in PAIRS[:4].tolist()])


class SkipGram(torch.nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.W = torch.nn.Embedding(vocab_size, dim)     # center-word vectors
        self.C = torch.nn.Embedding(vocab_size, dim)     # context-word vectors
        torch.nn.init.uniform_(self.W.weight, -0.5 / dim, 0.5 / dim)   # word2vec's initialization:
        torch.nn.init.zeros_(self.C.weight)                            # small W, zero C

    def forward(self, center, positive, negatives):      # (N,), (N,), (N, k)
        w = self.W(center)                               # (N, d)
        positive_score = (w * self.C(positive)).sum(-1)                       # (N,)
        negative_score = torch.einsum("nd,nkd->nk", w, self.C(negatives))     # (N, k)
        return -(F.logsigmoid(positive_score) + F.logsigmoid(-negative_score).sum(-1)).mean()


def train_skipgram(dim=16, k=5, epochs=15, batch_size=256, lr=0.01, seed=0):
    generator = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = SkipGram(len(VOCAB), dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(PAIRS), generator=generator)
        total = 0.0
        for start in range(0, len(PAIRS), batch_size):
            batch = PAIRS[order[start:start + batch_size]]
            negatives = torch.multinomial(NOISE, len(batch) * k, replacement=True, generator=generator).view(len(batch), k)
            loss = model(batch[:, 0], batch[:, 1], negatives)
            if epoch == 0 and start == 0:
                print(f"first minibatch: loss = {loss.item():.4f}")
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(batch)
        history.append(total / len(PAIRS))
        print(f"epoch {epoch + 1}: mean loss = {history[-1]:.4f}")
    return model, history


skipgram, p02_history = train_skipgram()

With five negatives and a zero context table, the first minibatch has loss $6\log 2=4.1589$. Positive and noise distributions overlap, so do not expect every sampled binary classification to be perfectly separable.

### Inspect the vectors

Normalize nonzero vectors and use their dot product as cosine similarity. The analogy query below is a diagnostic, not a language-understanding evaluation.

---

<!-- bilingual-zh -->

5 个负样本且上下文表初始为零时，第一个微批损失为 $6\log2=4.1589$。正样本与噪声分布重叠，不应期待每次采样二分类都完全可分。

### 检查向量
将非零向量归一化，以点积作为余弦相似度。下方类比查询是诊断工具，不是语言理解评估。


In [ ]:
VECTORS = F.normalize(skipgram.W.weight.detach(), dim=1)        # unit length, so a dot product is a cosine


def nearest(word, k=3):
    scores = VECTORS @ VECTORS[WORD_TO_ID[word]]
    scores[WORD_TO_ID[word]] = -1.0
    return [(VOCAB[i], round(scores[i].item(), 2)) for i in scores.topk(k).indices.tolist()]


def analogy(a, a_star, b, k=3):
    # a is to a_star as b is to ?
    query = F.normalize(VECTORS[WORD_TO_ID[a_star]] - VECTORS[WORD_TO_ID[a]] + VECTORS[WORD_TO_ID[b]], dim=0)
    scores = VECTORS @ query
    for word in (a, a_star, b):
        scores[WORD_TO_ID[word]] = -1.0
    return [VOCAB[i] for i in scores.topk(k).indices.tolist()]


for word in ["king", "cat", "rice", "paris"]:
    print(f"{word:6s} ->", nearest(word))
print("man : woman = king : ?   ", analogy("man", "woman", "king"))
print("man : woman = prince : ? ", analogy("man", "woman", "prince"))

The templates give groups distinguishable contexts, so recovering these groups is an intended result of the data construction. Try changing the seed or dimension. Which neighbors stay stable? For word types with identical context distributions, this objective has no contextual evidence to distinguish their meanings.

---

<!-- bilingual-zh -->

模板为各组提供可区分上下文，因此恢复这些分组是数据构造预期的结果。尝试改变随机种子或维度，观察哪些邻居保持稳定。对于上下文分布相同的词类型，这个目标没有上下文证据区分其含义。


## P03 · Verify one model row (optional course contribution)

Work in pairs on the [model-table issue #154](https://github.com/baojian/llm-26-fall/issues/154), then ask another pair to check your evidence and arithmetic. Follow that issue's scope for release dates and context lengths. Keep **Sources and Notes** as one column. This is an optional contribution to the course, not a new grading requirement.

The next cell prints the pinned official sources already recorded for E06. Reproducing the source check requires network access *outside this offline notebook run*: download the linked `config.json` and `tokenizer.json` only, verify their SHA-256 hashes, and inspect the fields below. Model weights are unnecessary. Do not infer unavailable artifacts for closed models.

1. Read `vocab_size`, `hidden_size`, and `tie_word_embeddings` from the configuration.
2. Count the union of IDs in `tokenizer['model']['vocab'].values()` and `entry['id']` for `entry` in `tokenizer['added_tokens']`.
3. Record the maximum ID and check that it fits the table.
4. Compute unique table parameters and bytes; state the dtype and whether weights are tied.
5. Include the model repository, immutable revision, source URLs, and any distinction between reported and derived values.

The counting method above applies to these JSON BPE tokenizers. Other tokenizer formats require their own inspection; do not assume every model publishes the same file layout.

---

<!-- bilingual-zh -->

## P03 · 核验模型表的一行（可选课程贡献）
根据原文链接的 issue #154 两人合作，再请另一组复核证据和计算；发布日期与上下文长度遵循该 issue 的范围，**Sources and Notes** 保持一列。这是可选贡献，不新增评分要求。

下一单元输出 E06 已记录的固定官方来源。若要复核，需在离线 notebook 运行之外联网，仅下载链接的 `config.json` 和 `tokenizer.json`，验证 SHA-256 并检查字段；不需要模型权重。不要推断闭源模型未公开的文件。

1. 从配置读取 `vocab_size`、`hidden_size`、`tie_word_embeddings`。
2. 对词表 ID 与 `added_tokens` ID 取并集。
3. 记录最大 ID，检查是否能放入表中。
4. 计算唯一表参数量和字节数，注明数据类型及是否绑定。
5. 记录模型仓库、不可变版本、来源 URL，并区分报告值与推导值。

该计数方法适用于这些 JSON BPE 词元化器；其他格式需要分别检查，不能假设文件结构相同。


In [ ]:
for cfg in model_evidence['models']:
    print(cfg['model'], '| revision:', cfg['revision'])
    for kind, evidence in cfg['sources'].items():
        print(kind, evidence['url'])
        print('SHA-256:', evidence['sha256'])

### Further reading

- [CS336 Lecture 2](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py): tensors, memory, gradients, and the training loop.
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html): raw logits and integer targets.
- [Bengio et al. (2003), Section 2](https://www.jmlr.org/papers/v3/bengio03a.html): a context network over learned word vectors.
- [Press and Wolf (2017)](https://aclanthology.org/E17-2025/): tied input/output embeddings.
- [Qwen3-Embedding report](https://arxiv.org/abs/2506.05176): representations trained for retrieval, distinct from raw token lookups.

The course deck links to `classical-reading.md` for optional background on counting and static embeddings.

---

<!-- bilingual-zh -->

### 延伸阅读
原文链接依次介绍 CS336 的张量、内存、梯度和训练循环；PyTorch 交叉熵的原始 logits 和整数目标；Bengio 等的词向量上下文网络；输入／输出嵌入绑定；以及用于检索、不同于原始词元查表的 Qwen3-Embedding 表示。课件的 `classical-reading.md` 提供计数和静态嵌入的可选背景。
